# Beag NIST Compliance Model Training

Train a domain-specific LoRA adapter that maps documents to NIST 800-53 / CSF / CMMC controls.

**Hardware**: Kaggle T4 GPU (16 GB VRAM)  
**Base model**: Qwen2.5-7B-Instruct (4-bit QLoRA)  
**Training recipe**: CISPO loss + interleaved batching + OPD recovery  
**Runtime**: ~15-25 min (200 examples + training)

## 0. Setup

In [ ]:
import os, sys, json, time
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/beag-training.git"  # TODO: replace with actual repo URL
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "beag-training"

In [ ]:
# GPU check
# Check for GPU
import subprocess
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("WARNING: No GPU detected. Training requires a GPU (T4 or higher).")


In [ ]:
# Clone the repo
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned.")

%cd {REPO_DIR}

In [ ]:
# Install Unsloth first (it pins specific torch/xformers versions)
# Upgrade build tools first, then install Unsloth (pins torch/xformers versions)
!pip install -q --upgrade pip setuptools wheel
!pip install -q unsloth

# Install remaining deps
# Install remaining deps (non-editable, to avoid build issues on Kaggle)
!pip install -q ".[embeddings]"


## 1. Generate synthetic training data

In [ ]:
# Provide your DeepSeek API key via Kaggle Secrets or paste it here
# Use Kaggle Secrets: Add-on → Secrets → DEEPSEEK_API_KEY
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["DEEPSEEK_API_KEY"] = user_secrets.get_secret("DEEPSEEK_API_KEY")
except Exception:
    # Fallback: enter key manually (not recommended, avoid committing)
    if not os.environ.get("DEEPSEEK_API_KEY"):
        print("Set DEEPSEEK_API_KEY via Kaggle Secrets or paste it below.")
        # Uncomment to enter manually:
        # os.environ["DEEPSEEK_API_KEY"] = "sk-..."

In [ ]:
NUM_EXAMPLES = 200      # 200 takes ~2 min; adjust up to 500 for better quality
CONCURRENCY = 10        # simultaneous API calls
AMBIGUITY_PROB = 0.5    # higher = more edge-case examples

from data import DataGenerator, GeneratorConfig
from data.validator import filter_valid
from data.augment import DataAugmenter
from frameworks.catalog import Framework, load_catalog

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

cat = load_catalog()
config = GeneratorConfig(
    total_examples=NUM_EXAMPLES,
    frameworks=list(Framework),
    concurrency=CONCURRENCY,
    max_mappings_per_example=6,
    min_mappings_per_example=3,
    ambiguity_prob=AMBIGUITY_PROB,
)
gen = DataGenerator(catalog=cat, config=config)

print(f"Generating {NUM_EXAMPLES} synthetic NIST compliance examples...")
t0 = time.monotonic()
import asyncio

async def _gen():
    return await gen.generate_all_async()

examples = asyncio.run(_gen())
examples, invalid = filter_valid(examples, cat)
print(f"  Valid: {len(examples)}  Invalid: {len(invalid)}  Time: {time.monotonic()-t0:.0f}s")

# Augment ~20%
augmenter = DataAugmenter(cat)
augmented = []
for ex in examples[:max(1, len(examples)//5)]:
    augmented.extend(augmenter.augment(ex))
aug_valid, _ = filter_valid(augmented, cat)
all_examples = examples + aug_valid

# Save
train_path = output_dir / "generated_augmented.jsonl"
with open(train_path, "w") as f:
    for ex in all_examples:
        record = {
            "text": json.dumps(ex.input_json),
            "mappings": [m.to_dict() for m in ex.mappings],
            "text_context": ex.text_context,
        }
        f.write(json.dumps(record) + "\n")

print(f"  Saved {len(all_examples)} examples to {train_path}")
print(f"  (Generated {len(examples)} + augmented {len(aug_valid)})")

## 2. Train with QLoRA + CISPO

In [ ]:
import torch
from onprem.unsloth_adapter import UnslothAdapter
from onprem.train import OnPremTrainer, load_dataset_from_jsonl

MODEL_TIER = "standard"    # Qwen2.5-7B-Instruct (~5 GB VRAM in 4-bit)
LORA_RANK = 32
BATCH_SIZE = 4
EPOCHS = 3
MAX_SEQ_LENGTH = 32768
LEARNING_RATE = 2e-5

print(f"Model tier: {MODEL_TIER}")
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Load base model with QLoRA
adapter = UnslothAdapter(
    model_tier=MODEL_TIER,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    lora_rank=LORA_RANK,
    lora_alpha=16,
)
model, tokenizer = adapter.load()
print(f"Model loaded: {adapter.model_id}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Load training data
records = load_dataset_from_jsonl(str(train_path))
print(f"Training records: {len(records)}")
print(f"  Has mappings: {sum(1 for r in records if 'mappings' in r)}")

trainer_config = {
    "batch_size": BATCH_SIZE,
    "num_epochs": EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "gradient_accumulation": 4,
    "warmup_steps": 50,
    "max_steps": 500,
    "output_dir": str(output_dir / "checkpoints"),
    "task_type": "nist_multi_label",
    "catalog": cat,
}

In [ ]:
print(f"Training {len(records)} examples, {EPOCHS} epochs, batch size {BATCH_SIZE}...")
print(f"  Effective batch: {BATCH_SIZE * trainer_config['gradient_accumulation']}")

trainer = OnPremTrainer(model, tokenizer, config=trainer_config)
t0 = time.monotonic()
result = trainer.train(records)
elapsed = time.monotonic() - t0

print(f"\nTraining complete in {elapsed:.0f}s")
print(f"  Steps: {result['total_steps']}")
print(f"  Best loss: {result['best_loss']:.4f}")
print(f"  Final loss: {result['final_loss']:.4f}")

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

history = result.get("history", {})
if history.get("loss"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.plot(history["loss"])
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Batch")
    ax1.set_ylabel("Loss")
    
    ax2.plot(history["lr"])
    ax2.set_title("Learning Rate (cosine warmup)")
    ax2.set_xlabel("Batch")
    ax2.set_ylabel("LR")
    
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=100)
    plt.show()

## 3. OPD Recovery (on-policy distillation)

In [ ]:
# Skip if your prompts file isn't available
# Load a teacher model (same base, frozen) to prevent instruction drift

ENABLE_OPD = False          # Set to True if you have prompts data
OPD_PROMPTS_PATH = "/kaggle/input/opd-prompts/tulu3-prompts.jsonl"

if ENABLE_OPD:
    from onprem.recipe import OPDRecovery, load_prompts_from_file
    
    if Path(OPD_PROMPTS_PATH).exists():
        teacher_adapter = UnslothAdapter(
            model_tier=MODEL_TIER,
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=True,
        )
        teacher_model, _ = teacher_adapter.load()
        teacher_model.eval()
        
        prompts = load_prompts_from_file(OPD_PROMPTS_PATH)
        print(f"OPD recovery with {len(prompts)} prompts...")
        
        opd = OPDRecovery(model, teacher_model, tokenizer, {
            "opd_steps": 50,
            "opd_batch_size": 2,
            "opd_max_tokens": 512,
            "opd_lr": 5e-6,
            "promote_every": 10,
            "temperature": 0.7,
        })
        opd_result = opd.recover(prompts)
        
        print(f"  OPD steps: {opd_result['opd_steps']}")
        print(f"  Final KL: {opd_result['final_loss']:.4f}")
        
        teacher_adapter.unload()
        del teacher_model, teacher_adapter
        torch.cuda.empty_cache()
    else:
        print(f"OPD prompts not found at {OPD_PROMPTS_PATH}, skipping.")
else:
    print("OPD recovery disabled (set ENABLE_OPD=True to enable).")

## 4. Save model

In [ ]:
# Save LoRA adapter (small, ~50-100 MB)
adapter_dir = WORK_DIR / "beag-nist-adapter"
adapter.save_adapter(str(adapter_dir))

# Save training summary
summary = {
    "base_model": adapter.model_id,
    "tier": MODEL_TIER,
    "lora_rank": LORA_RANK,
    "examples": len(records),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_loss": result["best_loss"],
    "final_loss": result["final_loss"],
    "total_steps": result["total_steps"],
    "training_time_s": elapsed,
}
with open(adapter_dir / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Adapter saved to: {adapter_dir}")
print(f"Size: {sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6:.1f} MB")
print("\nTraining complete. Download the adapter from the Output panel.")

## 5. Quick inference test

In [ ]:
from unsloth import FastLanguageModel

# Reload model for inference
model_inf, tok_inf = FastLanguageModel.from_pretrained(
    model_name=str(adapter_dir),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_inf)

test_doc = json.dumps({
    "type": "policy",
    "title": "Access Control Policy",
    "text": "All users must authenticate via multi-factor authentication before accessing protected systems.",
})

messages = [
    {"role": "system", "content": "You map policy documents to NIST 800-53 controls. Return a JSON array of control mappings with control_id, framework, confidence, and reasoning."},
    {"role": "user", "content": f"Map this document to relevant controls:\n\n{test_doc}"},
]

inputs = tok_inf.apply_chat_template(messages, tokenize=True, return_tensors="pt").to("cuda")
outputs = model_inf.generate(inputs, max_new_tokens=512, temperature=0.1)
response = tok_inf.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("Model prediction:")
try:
    print(json.dumps(json.loads(response), indent=2))
except json.JSONDecodeError:
    print(response)